In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from collections import Counter
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
warnings.filterwarnings("ignore")

In [2]:
# Load data
df = pd.read_csv("train.csv")

In [3]:
# normalize price
df["price"] = df["price"] / 100_000

In [14]:
# Fill brand
brand_list = df["brand"].value_counts().index.tolist()
UNKNOWN_BRAND_TAG = "unknown"
if not UNKNOWN_BRAND_TAG in [b.lower() for b in brand_list if isinstance(b, str)]:
    df["brand"] = df["brand"].fillna(UNKNOWN_BRAND_TAG)

In [15]:
df_model_price_variations = (
    df.groupby("modelId", as_index=False)
      .agg(
          prices=("price", lambda x: sorted(x.unique())),
          unique_price_count=("price", "nunique"),
      )
      .sort_values("unique_price_count", ascending=False)
      .reset_index(drop=True)
)

In [16]:
df_model_price_variations.head(4)

,modelId,prices,unique_price_count
0,225798114762,"[159.0, 190.0, 193.0, 199.0, 205.0]",5
1,240069696046,"[42.0, 55.0, 118.0, 120.0]",4
2,237155424280,"[530.0, 535.0, 550.0, 580.0]",4
3,237155424279,"[275.0, 280.0, 295.0, 325.0]",4


In [17]:
df_single_price_model = (
    df_model_price_variations[
        df_model_price_variations["unique_price_count"] == 1
    ]
    .reset_index(drop=True)
)

df_multiple_price_model = (
    df_model_price_variations[
        df_model_price_variations["unique_price_count"] > 1
    ]
    .reset_index(drop=True)
)

model_ids_single_price = df_single_price_model["modelId"].tolist()
model_ids_multiple_price = df_multiple_price_model["modelId"].tolist()

In [18]:
df_multiple_price_model.head(4)

,modelId,prices,unique_price_count
0,225798114762,"[159.0, 190.0, 193.0, 199.0, 205.0]",5
1,240069696046,"[42.0, 55.0, 118.0, 120.0]",4
2,237155424280,"[530.0, 535.0, 550.0, 580.0]",4
3,237155424279,"[275.0, 280.0, 295.0, 325.0]",4


In [19]:
brand_list = df["brand"].value_counts().index.tolist()
cat_ids = df["cat_id"].value_counts().index.tolist()

df_single_price = (
    df[df["modelId"].isin(model_ids_single_price)]
    .reset_index(drop=True)
)

df_multiple_price = (
    df[df["modelId"].isin(model_ids_multiple_price)]
    .reset_index(drop=True)
)

df_single_price_feature_variations = (
    df_single_price
    .drop(columns=["capturedAt"])
    .groupby("modelId")
    .nunique()
    .reset_index()
)

df_multiple_price_feature_variations = (
    df_multiple_price
    .drop(columns=["capturedAt"])
    .groupby("modelId")
    .nunique()
    .reset_index()
)

In [20]:
df_multiple_price_feature_variations["brand"].value_counts()

brand
1    1025
2       8
Name: count, dtype: int64

In [23]:
results = []

for modelId in df_multiple_price_feature_variations.loc[
    df_multiple_price_feature_variations["brand"] > 1, "modelId"
]:
    df_model = df[df["modelId"] == modelId]

    results.append({
        "modelId": modelId,
        "brand_counts": df_model["brand"].value_counts(dropna=False).to_dict()
    })

df_results = pd.DataFrame(results)

In [24]:
df_results

,modelId,brand_counts
0,69727504661,"{'unknown': 27, '滿漢大餐': 1}"
1,69727504662,"{'unknown': 27, '滿漢大餐': 1}"
2,79377014139,"{'unknown': 27, '滿漢大餐': 1}"
3,187208570747,"{'unknown': 27, '滿漢大餐': 1}"
4,240069696046,"{'unknown': 27, '滿漢大餐': 1}"
5,245272411578,"{'unknown': 27, '滿漢大餐': 1}"
6,256367054674,"{'unknown': 27, '滿漢大餐': 1}"
7,256837343723,"{'unknown': 27, '滿漢大餐': 1}"


In [25]:
df_multiple_price_feature_variations["cat_id"].value_counts()

cat_id
1    1009
2      24
Name: count, dtype: int64

In [26]:
results = []

for modelId in df_multiple_price_feature_variations.loc[
    df_multiple_price_feature_variations["cat_id"] > 1, "modelId"
]:
    df_model = df[df["modelId"] == modelId]

    results.append({
        "modelId": modelId,
        "cat_id_counts": df_model["cat_id"].value_counts(dropna=False).to_dict()
    })

df_results = pd.DataFrame(results)

In [27]:
df_results

,modelId,cat_id_counts
0,33786353474,"{100001: 10, 100630: 1}"
1,33786353475,"{100001: 10, 100630: 1}"
2,75936424827,"{100641: 111, 100637: 30}"
3,75936424828,"{100641: 111, 100637: 30}"
4,75936424829,"{100641: 111, 100637: 30}"
5,75936424830,"{100641: 111, 100637: 30}"
6,75936424831,"{100641: 111, 100637: 30}"
7,75936424832,"{100641: 111, 100637: 30}"
8,75936424833,"{100641: 111, 100637: 30}"
9,75936424834,"{100641: 111, 100637: 30}"


In [28]:
df_model = df[df["modelId"] == 33786353474]
df_model[["cat_id", "price"]]

,cat_id,price
13959,100001,149.0
14918,100001,149.0
24276,100001,149.0
27602,100001,149.0
35669,100001,149.0
36679,100001,149.0
40790,100001,89.0
43762,100001,89.0
49955,100001,89.0
61348,100001,89.0
